<a href="https://colab.research.google.com/github/maggiecope/comp351-ai-project/blob/main/Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler # Added
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
results = {}

In [ ]:
weather = pd.read_csv("weather.csv")
waits   = pd.read_csv("disney_wait_times.csv")

In [ ]:
# Clean waits data and extract time/date features
waits["Local Time"] = pd.to_datetime(waits["Local Time"], utc=True, errors="coerce")
waits = waits.dropna(subset=["Local Time"])
waits["Date"] = waits["Local Time"].dt.strftime("%Y-%m-%d")

In [ ]:
# Feature Engineering: Time-based features
waits["Hour"] = waits["Local Time"].dt.hour
waits["Minute"] = waits["Local Time"].dt.minute
waits["Is_Weekend"] = waits["Day of Week"].isin(["Saturday", "Sunday"]).astype(int)

In [ ]:
# Prepare weather data for merge
weather_for_merge = weather.rename(columns={"Day of datetime": "Date"})
weather_features = [
    "Date", "tempmax", "tempmin", "temp", "humidity", "precip", "windgust", "windspeed",
    "winddir", "sealevelpressure", "cloudcover", "visibility", "solarradiation",
    "solarenergy", "uvindex", "severerisk", "moonphase", "conditions"
]
weather_for_merge = weather_for_merge[weather_features]

In [ ]:
# Merge data and filter out zero wait times (which often represent closed rides or no data)
merged = waits.merge(weather_for_merge, on="Date", how="left")
merged = merged[merged["Wait Time"] > 0].copy()

In [ ]:
def categorize_time_of_day(hour):
    if 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

merged["Time_of_Day"] = merged["Hour"].apply(categorize_time_of_day)
# Optional: Temp Category feature (using temp)
merged["Temp_Category"] = pd.cut(
    merged["temp"],
    bins=[0, 65, 75, 85, 100],
    labels=["Cool", "Comfortable", "Warm", "Hot"],
    right=False
)

In [ ]:
# --- 3. Encode Categorical Features (Using Ordinal/Label Encoding) ---
merged["Land_Encoded"] = merged["Land"].astype("category").cat.codes
merged["Ride_Encoded"] = merged["Ride"].astype("category").cat.codes
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
merged["Day_of_Week_Encoded"] = merged["Day of Week"].map({day: i for i, day in enumerate(day_order)})
time_order = ["Morning", "Afternoon", "Evening", "Night"]
merged["Time_of_Day_Encoded"] = merged["Time_of_Day"].map({time: i for i, time in enumerate(time_order)})
merged["Conditions_Encoded"] = merged["conditions"].astype("category").cat.codes

In [ ]:
# --- 4. Prepare X and y, Split, and Scale ---
feature_cols = [
    "Land_Encoded", "Ride_Encoded", "Day_of_Week_Encoded",
    "Time_of_Day_Encoded", "Conditions_Encoded",
    "Hour", "Minute", "Is_Weekend",
    "tempmax", "tempmin", "temp",
    "humidity", "precip",
    "windgust", "windspeed", "winddir",
    "sealevelpressure",
    "cloudcover", "visibility",
    "solarradiation", "solarenergy",
    "uvindex", "severerisk",
    "moonphase"
]

X = merged[feature_cols]
y = merged["Wait Time"]

In [ ]:
# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale data (necessary for distance-based models like KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Data preparation complete. Training samples: {len(X_train):,} | Testing samples: {len(X_test):,}")

Data preparation complete. Training samples: 271,251 | Testing samples: 67,813


In [ ]:
# 1. Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)
y_pred_test_lin = lin_reg.predict(X_test_scaled)
lin_results = {
    'Test R²': r2_score(y_test, y_pred_test_lin),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test_lin)),
    'Test MAE': mean_absolute_error(y_test, y_pred_test_lin)
}
results['Linear Regression'] = lin_results
print("1. Linear Regression trained.")
print(f"  Test R²:    {lin_results['Test R²']:.4f}")
print(f"  Test RMSE:  {lin_results['Test RMSE']:.2f}")
print(f"  Test MAE:   {lin_results['Test MAE']:.2f} minutes\n")

1. Linear Regression trained.
  Test R²:    0.1980
  Test RMSE:  22.13
  Test MAE:   16.91 minutes



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sort results by R² for better visualization
results_df_sorted = results_df.sort_values(by='Test R²', ascending=False)

# Plotting R² Score
plt.figure(figsize=(12, 6))
sns.barplot(x=results_df_sorted.index, y='Test R²', data=results_df_sorted, palette='viridis')
plt.title('Model Performance: Test R² Score')
plt.xlabel('Model')
plt.ylabel('R² Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


NameError: name 'results_df' is not defined

In [ ]:
# Plotting RMSE
plt.figure(figsize=(12, 6))
sns.barplot(x=results_df_sorted.index, y='Test RMSE', data=results_df_sorted, palette='magma')
plt.title('Model Performance: Test RMSE')
plt.xlabel('Model')
plt.ylabel('RMSE (minutes)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Plotting MAE
plt.figure(figsize=(12, 6))
sns.barplot(x=results_df_sorted.index, y='Test MAE', data=results_df_sorted, palette='plasma')
plt.title('Model Performance: Test MAE')
plt.xlabel('Model')
plt.ylabel('MAE (minutes)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# 2. Polynomial Regression (Degree 2)
# Transform scaled features into polynomial features
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

poly_reg = LinearRegression()
poly_reg.fit(X_train_poly, y_train)
y_pred_test_poly = poly_reg.predict(X_test_poly)
poly_results = {
    'Test R²': r2_score(y_test, y_pred_test_poly),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test_poly)),
    'Test MAE': mean_absolute_error(y_test, y_pred_test_poly)
}
results['Polynomial Regression'] = poly_results
print("2. Polynomial Regression (Degree 2) trained.")
print(f"  Test R²:    {poly_results['Test R²']:.4f}")
print(f"  Test RMSE:  {poly_results['Test RMSE']:.2f}")
print(f"  Test MAE:   {poly_results['Test MAE']:.2f} minutes\n")

In [ ]:
# 3. K-Neighbors Regressor (Requires Scaled Data)
knn_reg = KNeighborsRegressor(n_neighbors=5, n_jobs=-1)
knn_reg.fit(X_train_scaled, y_train)
y_pred_test_knn = knn_reg.predict(X_test_scaled)
knn_results = {
    'Test R²': r2_score(y_test, y_pred_test_knn),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test_knn)),
    'Test MAE': mean_absolute_error(y_test, y_pred_test_knn)
}
results['K-Neighbors'] = knn_results
print("3. K-Neighbors Regressor trained.")
print(f"  Test R²:    {knn_results['Test R²']:.4f}")
print(f"  Test RMSE:  {knn_results['Test RMSE']:.2f}")
print(f"  Test MAE:   {knn_results['Test MAE']:.2f} minutes\n")

In [ ]:
# 4. Support Vector Regressor (SVR)
svr_reg = SVR(kernel='rbf', C=10, epsilon=0.1)
svr_reg.fit(X_train_scaled, y_train)
y_pred_test_svr = svr_reg.predict(X_test_scaled)
svr_results = {
    'Test R²': r2_score(y_test, y_pred_test_svr),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test_svr)),
    'Test MAE': mean_absolute_error(y_test, y_pred_test_svr)
}
results['SVR'] = svr_results
print("4. Support Vector Regressor (SVR) trained.")
print(f"  Test R²:    {svr_results['Test R²']:.4f}")
print(f"  Test RMSE:  {svr_results['Test RMSE']:.2f}")
print(f"  Test MAE:   {svr_results['Test MAE']:.2f} minutes\n")

In [ ]:
# 5. ANN
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(80, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.1),
    layers.Dense(40, activation='relu'),
    layers.Dense(1)
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

print("\nModel Architecture:")
model.summary()

In [ ]:
# Train the model

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
y_pred = model.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\nTest Set Performance:")
print(f"Mean Absolute Error (MAE): {mae:.2f} minutes")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} minutes")
print(f"R² Score: {r2:.4f}")

In [ ]:
# 6. Decision Tree Regressor
dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train, y_train)
y_pred_test_dt = dt_reg.predict(X_test)
dt_results = {
    'Test R²': r2_score(y_test, y_pred_test_dt),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test_dt)),
    'Test MAE': mean_absolute_error(y_test, y_pred_test_dt)
}
results['Decision Tree'] = dt_results
print("6. Decision Tree Regressor trained.")
print(f"  Test R²:    {dt_results['Test R²']:.4f}")
print(f"  Test RMSE:  {dt_results['Test RMSE']:.2f}")
print(f"  Test MAE:   {dt_results['Test MAE']:.2f} minutes\n")

In [ ]:
# 7. Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train)
y_pred_test_rf = rf_reg.predict(X_test)
rf_results = {
    'Test R²': r2_score(y_test, y_pred_test_rf),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test_rf)),
    'Test MAE': mean_absolute_error(y_test, y_pred_test_rf)
}
results['Random Forest'] = rf_results
print("7. Random Forest Regressor trained.")
print(f"  Test R²:    {rf_results['Test R²']:.4f}")
print(f"  Test RMSE:  {rf_results['Test RMSE']:.2f}")
print(f"  Test MAE:   {rf_results['Test MAE']:.2f} minutes\n")

In [ ]:
# 8. Gradient Boosting Regressor
gbr_reg = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
gbr_reg.fit(X_train, y_train)
y_pred_test_gbr = gbr_reg.predict(X_test)
gbr_results = {
    'Test R²': r2_score(y_test, y_pred_test_gbr),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test_gbr)),
    'Test MAE': mean_absolute_error(y_test, y_pred_test_gbr)
}
results['Gradient Boosting'] = gbr_results
print("8. Gradient Boosting Regressor trained.")
print(f"  Test R²:    {gbr_results['Test R²']:.4f}")
print(f"  Test RMSE:  {gbr_results['Test RMSE']:.2f}")
print(f"  Test MAE:   {gbr_results['Test MAE']:.2f} minutes\n")

In [ ]:
# --- CONCLUDE AND DISPLAY RESULTS ---
results_df = pd.DataFrame(results).T
print("\nModel Performance Comparison (Test Set Metrics):")
print("-------------------------------------------------")
print(results_df[['Test R²', 'Test RMSE', 'Test MAE']].sort_values(by='Test R²', ascending=False).to_markdown(floatfmt=".4f"))